# 02. Modeling — ESS 배터리 수명 예측 (DAY 2)
DAY 1 전략 : log10(수명) 타깃 + ΔQ 분산(log) 핵심 피처 + 규제 선형 회귀 주력

## Step 0. 피처 표 만들기
- 셀 단위 피처 16개 (2~100번째 사이클) + ΔQ 피처 + 충전 피처
- IR 미측정 6개(Batch2_41~46)는 빼지 않는다. (피처가 비었다는 이유로 테스트 배터리를 빼면 평가 조건이 달라지기 때문이다. IR 피처는 Q5에서 사용하지 않기로 했으므로, IR이 측정되지 않아도 문제가 없다.)

In [2]:
# Step 0. 셀 단위 피처 표 만들기 — IR이 없는 배터리도 빠지지 않게
import numpy as np
import pandas as pd

clean = pd.read_pickle("../data/cells_clean.pkl")  # 정제 후 119셀
summary_df = pd.read_pickle("../data/summary_df.pkl")
dq_df = pd.read_pickle("../data/dq_features.pkl")
pol = pd.read_pickle("../data/policy_features.pkl")

early = summary_df[
    summary_df["cycle"].between(2, 100) & summary_df["QD"].between(0.7, 1.2)
]


def early_feats(g):
    g = g.sort_values("cycle")
    ir = g[g["IR"] > 0]  # IR 조건은 IR 피처에만 적용 (배터리는 빼지 않음)
    return pd.Series(
        {
            "QD_2": g["QD"].iloc[0],
            "QD_diff": g["QD"].iloc[-1] - g["QD"].iloc[0],
            "QD_slope": np.polyfit(g["cycle"], g["QD"], 1)[0],
            "IR_2": ir["IR"].iloc[0] if len(ir) else np.nan,
            "IR_diff": ir["IR"].iloc[-1] - ir["IR"].iloc[0] if len(ir) else np.nan,
            "Tavg_mean": g["Tavg"].mean(),
            "Tmax_mean": g["Tmax"].mean(),
            "chargetime_med": g["chargetime"].iloc[:5].median(),
        }
    )


ef = early.groupby("cell_key").apply(early_feats).reset_index()

feat = (
    clean[["cell_key", "batch", "cycle_life"]]
    .merge(ef, on="cell_key")
    .merge(
        dq_df[["cell_key", "dq_min", "dq_mean", "dq_var", "dq_logvar"]], on="cell_key"
    )
    .merge(pol[["cell_key", "C1", "Q1", "C2", "Cavg_80"]], on="cell_key")
)
feat["log_life"] = np.log10(feat["cycle_life"])

# ✅ 개수 확인 : 정제 후 배치별 개수(36 / 39 / 44)와 같아야 함
print("정제 후 :", clean.groupby("batch").size().to_dict())
print("피처 표 :", feat.groupby("batch").size().to_dict())
assert (
    feat.groupby("batch").size().to_dict() == clean.groupby("batch").size().to_dict()
), "배터리 수가 다름!"
print("IR 비어 있는 배터리 :", feat.loc[feat["IR_2"].isna(), "cell_key"].tolist())

feat.to_pickle("../data/features.pkl")

정제 후 : {'Batch1': 36, 'Batch2': 39, 'Batch3': 44}
피처 표 : {'Batch1': 36, 'Batch2': 39, 'Batch3': 44}
IR 비어 있는 배터리 : ['Batch2_41', 'Batch2_42', 'Batch2_43', 'Batch2_44', 'Batch2_45', 'Batch2_46']


## Step 1. 데이터 나누기
- 학습 : Batch 1 (36개) 중 27개 / 검증 : Batch 1 중 9개 (수명 3구간 층화) / 테스트 : Batch 2 (39개, 마지막에 1번만)
- 층화 이유 : 검증용에 짧은·중간·긴 배터리가 골고루 들어가야 검증 점수를 믿을 수 있음

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42  # 재현성을 위해 모든 무작위 값 고정
feat = pd.read_pickle("../data/features.pkl")  # 어제 만든 셀 단위 피처 (정제 후 119셀)
b1 = feat[feat["batch"] == "Batch1"].reset_index(drop=True)  # 학습용 풀 36개
b2 = feat[feat["batch"] == "Batch2"].reset_index(
    drop=True
)  # 테스트 39개 (마지막에 1번만)

# 층화 : 수명을 3구간(짧음/중간/긺)으로 나누고, 구간마다 고르게 검증용 9개를 뽑음
life_bin = pd.qcut(b1["cycle_life"], q=3, labels=["short", "mid", "long"])
train_df, valid_df = train_test_split(
    b1, test_size=9, stratify=life_bin, random_state=RANDOM_STATE
)

print(f"train {len(train_df)}개 / valid {len(valid_df)}개 / test {len(b2)}개")
print(
    "train 수명 범위 :",
    int(train_df["cycle_life"].min()),
    "~",
    int(train_df["cycle_life"].max()),
)
print("valid 수명 9개  :", sorted(valid_df["cycle_life"].astype(int)))

train 27개 / valid 9개 / test 39개
train 수명 범위 : 534 ~ 1074
valid 수명 9개  : [599, 648, 651, 704, 709, 857, 870, 880, 1017]


## Step 2. 기준선 : ΔQ 분산(log) 1개 + 선형 회귀
- 타깃 : log10(수명) → 예측 후 10^ŷ로 되돌려 MAPE 계산
- Train 점수 : 학습 27개 안에서 교차검증(CV). 같은 충전 방식 배터리는 같은 묶음에 넣음 (누수 방지)
- Valid 점수 : 떼어 둔 9개 / Test(Batch 2)는 아직 쓰지 않음

In [4]:
# Step 2. 평가 함수 + 기준선 모델
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_predict


def mape(life_true, life_pred):
    return np.mean(np.abs(life_pred - life_true) / life_true) * 100


policy = pol.set_index("cell_key")["charging_policy"]  # 배터리별 충전 방식


def evaluate(model, cols, name):
    X_tr, y_tr = train_df[cols], train_df["log_life"]
    groups = train_df["cell_key"].map(policy)  # 같은 충전 방식 = 같은 묶음
    cv_pred = cross_val_predict(
        model, X_tr, y_tr, cv=GroupKFold(n_splits=5), groups=groups
    )
    cv_mape = mape(train_df["cycle_life"], 10**cv_pred)

    model.fit(X_tr, y_tr)  # 학습 27개 전체로 학습
    va_mape = mape(valid_df["cycle_life"], 10 ** model.predict(valid_df[cols]))
    return {
        "실험": name,
        "피처": ", ".join(cols),
        "Train CV MAPE": round(cv_mape, 1),
        "Valid MAPE": round(va_mape, 1),
    }


exp_log = []  # 실험 이력 표
exp_log.append(
    evaluate(
        make_pipeline(StandardScaler(), LinearRegression()),
        ["dq_logvar"],
        "E1 기준선 선형회귀",
    )
)
pd.DataFrame(exp_log)

,실험,피처,Train CV MAPE,Valid MAPE
0,E1 기준선 선형회귀,dq_logvar,10.5,7.1


#### 결과 해석 : 과적합은 없다. 다만 Valid 7.1%는 운과 쉬운 문제 덕을 봤을 수 있으니, 실력은 Train CV 10.5% 쪽이 더 현실적이라고 보는 게 안전하다.

## Step 3. 보조 피처 하나씩 추가 (선형 회귀)
- 후보 : 용량 감소 기울기(QD_slope), 80%까지 평균 충전 속도(Cavg_80), 평균 온도(Tavg_mean)
- 판단 : Train CV MAPE가 기준선(10.5%)보다 낮아질 때만 채택

In [7]:
# Step 3. 보조 피처 하나씩 추가
for extra in ["QD_slope", "Cavg_80", "Tavg_mean"]:
    exp_log.append(
        evaluate(
            make_pipeline(StandardScaler(), LinearRegression()),
            ["dq_logvar", extra],
            f"E2 + {extra}",
        )
    )
pd.DataFrame(exp_log).drop_duplicates("실험")

,실험,피처,Train CV MAPE,Valid MAPE
0,E1 기준선 선형회귀,dq_logvar,10.5,7.1
1,E2 + QD_slope,"dq_logvar, QD_slope",13.5,6.9
2,E2 + Cavg_80,"dq_logvar, Cavg_80",11.0,7.8
3,E2 + Tavg_mean,"dq_logvar, Tavg_mean",11.0,7.1


## Step 4. 규제 모델(Ridge·Lasso) + 트리 모델(RandomForest) 비교
- 피처 4개(dq_logvar, QD_slope, Cavg_80, Tavg_mean)를 한꺼번에 주고, 규제로 불필요한 피처를 누르게 함
- 규제 세기(α)는 학습 데이터 CV로 고름 (하이퍼파라미터가 α 하나뿐이라 넓은 격자 탐색으로 진행)
- RandomForest는 비교용 (학습 범위 밖 예측 불가 → 주력 아님)

In [8]:
# Step 4. 규제 모델 + 트리 모델 (튜닝은 학습 데이터 CV로만)
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

cols_all = ["dq_logvar", "QD_slope", "Cavg_80", "Tavg_mean"]
groups = train_df["cell_key"].map(policy)


def tune(estimator, grid):
    gs = GridSearchCV(
        make_pipeline(StandardScaler(), estimator),
        grid,
        cv=GroupKFold(n_splits=5),
        scoring="neg_mean_absolute_error",
    )
    gs.fit(train_df[cols_all], train_df["log_life"], groups=groups)
    return gs.best_estimator_, gs.best_params_


ridge, p_r = tune(Ridge(), {"ridge__alpha": np.logspace(-3, 3, 30)})
lasso, p_l = tune(Lasso(max_iter=10000), {"lasso__alpha": np.logspace(-4, 0, 30)})
rf, p_f = tune(
    RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    {
        "randomforestregressor__max_depth": [2, 3, 4, None],
        "randomforestregressor__min_samples_leaf": [2, 4],
    },
)
print("Ridge 최적 :", p_r)
print("Lasso 최적 :", p_l)
print("RF 최적    :", p_f)

for name, model in [
    ("E3 Ridge (피처 4개)", ridge),
    ("E4 Lasso (피처 4개)", lasso),
    ("E5 RandomForest (피처 4개)", rf),
]:
    exp_log.append(evaluate(model, cols_all, name))

# Lasso가 피처를 어떻게 골랐는지 : 계수가 0이면 "안 쓴다"는 뜻
print("\nLasso 계수 :", dict(zip(cols_all, lasso[-1].coef_.round(4))))
pd.DataFrame(exp_log).drop_duplicates("실험")

Ridge 최적 : {'ridge__alpha': np.float64(1.2689610031679222)}
Lasso 최적 : {'lasso__alpha': np.float64(0.008531678524172805)}
RF 최적    : {'randomforestregressor__max_depth': None, 'randomforestregressor__min_samples_leaf': 2}

Lasso 계수 : {'dq_logvar': np.float64(-0.0561), 'QD_slope': np.float64(0.0), 'Cavg_80': np.float64(-0.0066), 'Tavg_mean': np.float64(-0.0)}


,실험,피처,Train CV MAPE,Valid MAPE
0,E1 기준선 선형회귀,dq_logvar,10.5,7.1
1,E2 + QD_slope,"dq_logvar, QD_slope",13.5,6.9
2,E2 + Cavg_80,"dq_logvar, Cavg_80",11.0,7.8
3,E2 + Tavg_mean,"dq_logvar, Tavg_mean",11.0,7.1
10,E3 Ridge (피처 4개),"dq_logvar, QD_slope, Cavg_80, Tavg_mean",12.5,9.0
11,E4 Lasso (피처 4개),"dq_logvar, QD_slope, Cavg_80, Tavg_mean",11.6,8.1
12,E5 RandomForest (피처 4개),"dq_logvar, QD_slope, Cavg_80, Tavg_mean",13.2,7.2


## Step 5. 최종 모델로 테스트 (Batch 2, 1회)
- 최종 모델 : E1 선형 회귀 (dq_logvar 1개) — CV가 가장 좋고 가장 단순함
- 모델 선택이 끝났으므로 Batch 1 전체(36개)로 다시 학습한 뒤 Batch 2(39개)를 한 번만 예측

In [11]:
# Step 5. 최종 모델 → Batch 2 테스트 (1회)
from sklearn.metrics import mean_squared_error, r2_score

final = make_pipeline(StandardScaler(), LinearRegression())
final.fit(b1[["dq_logvar"]], b1["log_life"])  # Batch 1 전체 36개로 학습

test_pred = 10 ** final.predict(
    b2[["dq_logvar"]]
)  # Batch 2 39개 예측 (수명으로 되돌림)
test_mape = mape(b2["cycle_life"], test_pred)
test_rmse = np.sqrt(mean_squared_error(b2["cycle_life"], test_pred))
test_r2 = r2_score(b2["cycle_life"], test_pred)
bias = (
    np.mean((test_pred - b2["cycle_life"]) / b2["cycle_life"]) * 100
)  # (+)면 길게 예측

e1 = pd.DataFrame(exp_log).drop_duplicates("실험").iloc[0]
train_m, valid_m, target = e1["Train CV MAPE"], e1["Valid MAPE"], 9.1

perf = pd.DataFrame(
    {
        "MAPE (%)": [
            train_m,
            valid_m,
            round(test_mape, 1),
            round(valid_m - train_m, 1),
            round(test_mape - valid_m, 1),
            round(test_mape - target, 1),
        ]
    },
    index=[
        "Train (Batch 1 CV)",
        "Valid (Batch 1 Hold-out)",
        "Test (Batch 2)",
        "Gap (Train-Valid)",
        "Gap (Valid-Test)",
        "Gap (Target-Test)",
    ],
)
print(
    f"보조 지표 : RMSE {test_rmse:.0f} 사이클, R² {test_r2:.2f}, 평균 오차 방향 {bias:+.1f}%"
)
perf.to_csv("../results/model_performance.csv")
perf

보조 지표 : RMSE 163 사이클, R² 0.45, 평균 오차 방향 +28.1%


,MAPE (%)
Train (Batch 1 CV),10.5
Valid (Batch 1 Hold-out),7.1
Test (Batch 2),28.6
Gap (Train-Valid),-3.4
Gap (Valid-Test),21.5
Gap (Target-Test),19.5


#### 성능 해석
 Batch 2 수명을 실제보다 길게 예측했다(평균 +28%). 같은 ΔQ 분산이어도 Batch 2 배터리는 수명이 더 짧아서, Batch 1로 배운 관계가 Batch 2에는 높게 맞춰져 있기 때문이다.

## Step 6. 오류 분석 (모델은 고치지 않고 분석만)

In [12]:
# Step 6. 오차가 큰 배터리 살펴보기
err = b2[["cell_key", "cycle_life", "dq_logvar"]].copy()
err["pred"] = test_pred.round(0)
err["err_%"] = ((test_pred - b2["cycle_life"]) / b2["cycle_life"] * 100).round(1)
err["policy"] = err["cell_key"].map(policy)

print("길게 예측한 배터리 :", (err["err_%"] > 0).sum(), "/", len(err))
print("\n[오차가 가장 큰 5개]")
print(err.sort_values("err_%", key=abs, ascending=False).head(5).to_string(index=False))

# 수명 구간별 평균 오차
err["life_bin"] = pd.cut(
    err["cycle_life"], [0, 500, 800, 2000], labels=["~500", "500~800", "800~"]
)
print("\n[수명 구간별 평균 오차 %]")
print(err.groupby("life_bin", observed=True)["err_%"].agg(["count", "mean"]).round(1))

길게 예측한 배터리 : 38 / 39

[오차가 가장 큰 5개]
 cell_key  cycle_life  dq_logvar   pred  err_%                      policy
 Batch2_6       393.0  -3.530040  648.0   64.9                 3.6C(9%)-5C
Batch2_18       449.0  -3.707303  732.0   63.0             5.2C(50%)-4.25C
Batch2_15       396.0  -3.516252  642.0   62.1                 3.6C(9%)-5C
 Batch2_2       424.0  -3.460248  618.0   45.7             5.2C(50%)-4.25C
 Batch2_9       791.0  -4.344426 1132.0   43.1 5.6C(26%)-4.5C-newstructure

[수명 구간별 평균 오차 %]
          count  mean
life_bin             
~500         28  31.9
500~800       4  35.2
800~          7   9.1



#### 분석노트
- 공통점: 크게 틀린 배터리는 수명 500 미만의 짧은 배터리이며, 39개 중 38개를 실제보다 길게 예측했다. 수명 800 이상은 오차 9.1%로 원논문 수준이다.
- 원인 가설: 학습 데이터(Batch 1)의 최소 수명은 534로, 400대 배터리를 학습한 적이 없다. Batch 2의 짧은 배터리는 같은 ΔQ 값의 Batch 1 배터리보다 훨씬 빨리 수명이 끝나, Batch 1로 배운 관계로는 길게 예측된다.
- 기각한 가설: Batch 2에만 있는 "2단계가 더 빠른 충전 방식"이 원인인지 확인했으나, 해당 8개와 나머지의 평균 오차가 비슷해(+29.4% vs +27.8%) 원인이 아니었다.
- 개선 방향: 짧은 수명 배터리를 학습 데이터에 포함하기(예: 여러 배치를 섞어 학습), 배치 간 차이를 보정할 수 있는 피처 탐색.